### Import Packages

In [1]:
import duckdb
from pathlib import Path

#### Configs and Variables

In [2]:
data_vault_path = r"C:\Users\JacksonDivakarRajasi\OneDrive - JMAN Group Ltd\Documents\Python Folder\DuckDB Environment\Olist Eccomerce DuckDB\DuckDB Data\data_vault.duckdb"

#### DuckDB Connection

In [3]:
con = duckdb.connect(data_vault_path)

In [4]:
con.execute("""
CREATE SCHEMA IF NOT EXISTS mart;
""")

### Dim Customer

In [5]:
con.execute("""
CREATE OR REPLACE TABLE mart.dim_customer AS

WITH versions AS (
    SELECT
        c.customer_hk,
        h.customer_id,
        c.customer_unique_id,
        c.customer_zip_code_prefix,
        c.customer_city,
        c.customer_state,
        c.latitude,
        c.longitude,

        c.load_date AS effective_from,

        LEAD(c.load_date) OVER (
            PARTITION BY c.customer_hk
            ORDER BY c.load_date
        ) AS effective_to

    FROM business_vault.customer_enriched_history c

    JOIN raw_vault.hub_customer h
        ON c.customer_hk = h.customer_hk
)

SELECT
    ROW_NUMBER() OVER (
        ORDER BY customer_hk, effective_from
    ) AS customer_sk,

    *,
    
    CASE
        WHEN effective_to IS NULL THEN TRUE
        ELSE FALSE
    END AS is_current

FROM versions;
""")

### Dim Product

In [6]:
con.execute("""
CREATE OR REPLACE TABLE mart.dim_product AS

WITH versions AS (
    SELECT
        p.product_hk,
        h.product_id,

        p.product_category_name,
        p.product_category_name_english,
        p.product_name_length,
        p.product_description_length,
        p.product_photos_qty,
        p.product_weight_g,
        p.product_length_cm,
        p.product_height_cm,
        p.product_width_cm,

        p.load_date AS effective_from,

        LEAD(p.load_date) OVER (
            PARTITION BY p.product_hk
            ORDER BY p.load_date
        ) AS effective_to

    FROM business_vault.product_enriched_history p

    JOIN raw_vault.hub_product h
        ON p.product_hk = h.product_hk
)

SELECT
    ROW_NUMBER() OVER (
        ORDER BY product_hk, effective_from
    ) AS product_sk,

    *,

    CASE
        WHEN effective_to IS NULL THEN TRUE
        ELSE FALSE
    END AS is_current

FROM versions;
""")

### Dim Seller

In [7]:
con.execute("""
CREATE OR REPLACE TABLE mart.dim_seller AS

WITH versions AS (
    SELECT
        s.seller_hk,
        h.seller_id,

        s.seller_zip_code_prefix,
        s.seller_city,
        s.seller_state,
        s.latitude,
        s.longitude,

        s.load_date AS effective_from,

        LEAD(s.load_date) OVER (
            PARTITION BY s.seller_hk
            ORDER BY s.load_date
        ) AS effective_to

    FROM business_vault.seller_enriched_history s

    JOIN raw_vault.hub_seller h
        ON s.seller_hk = h.seller_hk
)

SELECT
    ROW_NUMBER() OVER (
        ORDER BY seller_hk, effective_from
    ) AS seller_sk,

    *,

    CASE
        WHEN effective_to IS NULL THEN TRUE
        ELSE FALSE
    END AS is_current

FROM versions;
""")

### Dim Date

In [8]:
con.execute("""
CREATE OR REPLACE TABLE mart.dim_date AS

WITH date_range AS (
    SELECT
        MIN(CAST(order_purchase_timestamp AS DATE)) AS min_date,
        MAX(CAST(order_estimated_delivery_date AS DATE)) AS max_date
    FROM raw_vault.sat_order_details
)

SELECT
    CAST(STRFTIME(d, '%Y%m%d') AS INTEGER) AS date_key,
    CAST(d AS DATE) AS full_date,

    YEAR(d) AS year,
    QUARTER(d) AS quarter,
    MONTH(d) AS month_number,
    MONTHNAME(d) AS month_name,
    DAY(d) AS day_of_month,
    DAYOFWEEK(d) AS day_of_week

FROM date_range,
GENERATE_SERIES(
    min_date,
    max_date,
    INTERVAL '1 day'
) t(d);
""")